# Model Compatibility with Flask and CoreML

This notebook demonstrates how to check if a model is compatible with Flask applications and Apple's CoreML format, and how to prepare it for deployment.

## Overview

When deploying AI models, it's important to ensure they're compatible with the target platforms. This notebook focuses on two common deployment targets:

1. **Flask Applications**: Web services that serve model predictions via HTTP APIs
2. **Apple's CoreML**: Framework for running machine learning models on Apple devices

We'll check model compatibility, identify potential issues, and make necessary adjustments to ensure smooth deployment.

In [ ]:
import os
import sys
import torch
import yaml
import logging
from pathlib import Path

# Add the project root to the path
project_root = Path().absolute().parent
sys.path.append(str(project_root))

# Import project modules
from src.model.architecture import TransformerModel
from src.model.compatibility_wrapper import CompatibilityWrapper
from src.utils.compatibility_checker import CompatibilityChecker

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## 1. Load Configuration and Create Model

First, let's load the configuration and create a model.

In [ ]:
# Load configuration
config_path = project_root / 'configs' / 'config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Extract model configuration
model_config = config['model']

# Create output directory
output_dir = project_root / 'outputs' / 'compatibility'
os.makedirs(output_dir, exist_ok=True)

print(f"Model configuration:\n{model_config}")

In [ ]:
# Create model
model = TransformerModel(
    vocab_size=model_config['vocab_size'],
    hidden_size=model_config['hidden_size'],
    num_hidden_layers=model_config['num_hidden_layers'],
    num_attention_heads=model_config['num_attention_heads'],
    intermediate_size=model_config['intermediate_size'],
    hidden_dropout_prob=model_config.get('hidden_dropout_prob', 0.1),
    attention_probs_dropout_prob=model_config.get('attention_probs_dropout_prob', 0.1),
    max_position_embeddings=model_config['max_position_embeddings'],
    initializer_range=model_config.get('initializer_range', 0.02)
)

# Set model to evaluation mode
model.eval()

print(f"Model created successfully")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Check Model Compatibility

Now, let's check if the model is compatible with Flask applications and Apple's CoreML format.

In [ ]:
# Create compatibility checker
checker = CompatibilityChecker(model, output_dir=str(output_dir))

# Check compatibility
compatibility_info = checker.check_compatibility()

# Print results
print(f"Flask compatibility: {compatibility_info['flask']['is_compatible']}")
print(f"CoreML compatibility: {compatibility_info['coreml']['is_compatible']}")
print(f"Overall compatibility: {compatibility_info['overall']['is_compatible']}")

## 3. Review Compatibility Issues

Let's review any compatibility issues and recommendations.

In [ ]:
# Review Flask compatibility issues
print("Flask Compatibility Issues:")
for issue in compatibility_info['flask']['issues']:
    print(f"- {issue}")

print("\nFlask Compatibility Recommendations:")
for rec in compatibility_info['flask']['recommendations']:
    print(f"- {rec}")

print("\nCoreML Compatibility Issues:")
for issue in compatibility_info['coreml']['issues']:
    print(f"- {issue}")

print("\nCoreML Compatibility Recommendations:")
for rec in compatibility_info['coreml']['recommendations']:
    print(f"- {rec}")

## 4. Prepare Model for Deployment

Now, let's prepare the model for deployment by wrapping it with a compatibility wrapper.

In [ ]:
# Wrap model with compatibility wrapper
wrapped_model = CompatibilityWrapper(model)

# Check compatibility of wrapped model
wrapped_checker = CompatibilityChecker(wrapped_model, output_dir=str(output_dir / 'wrapped'))
wrapped_compatibility_info = wrapped_checker.check_compatibility()

# Print results
print(f"Wrapped model Flask compatibility: {wrapped_compatibility_info['flask']['is_compatible']}")
print(f"Wrapped model CoreML compatibility: {wrapped_compatibility_info['coreml']['is_compatible']}")
print(f"Wrapped model overall compatibility: {wrapped_compatibility_info['overall']['is_compatible']}")

## 5. Save Model for Deployment

Let's save the model for deployment to both Flask applications and CoreML.

In [ ]:
# Save model for Flask deployment
flask_dir = output_dir / 'flask'
os.makedirs(flask_dir, exist_ok=True)

# Save model in PyTorch format
torch.save(wrapped_model.state_dict(), flask_dir / 'model.pt')

# Save model configuration
with open(flask_dir / 'config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

print(f"Model saved for Flask deployment in {flask_dir}")

In [ ]:
# Save model for CoreML conversion
coreml_dir = output_dir / 'coreml'
os.makedirs(coreml_dir, exist_ok=True)

# Prepare model for export
export_model = wrapped_model.prepare_for_export()

# Save model in TorchScript format
export_model.to_torchscript(str(coreml_dir / 'model.pt'))

# Save model configuration
with open(coreml_dir / 'config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

print(f"Model saved for CoreML conversion in {coreml_dir}")

## 6. Test Model Inference

Let's test the model inference to ensure it works correctly.

In [ ]:
# Create test input
input_ids = torch.randint(0, model_config['vocab_size'], (1, 10))
attention_mask = torch.ones_like(input_ids)

# Run inference with original model
with torch.no_grad():
    original_output = model(input_ids, attention_mask)

# Run inference with wrapped model
with torch.no_grad():
    wrapped_output = wrapped_model(input_ids, attention_mask)

# Compare outputs
print(f"Original output shape: {original_output.shape}")
print(f"Wrapped output shape: {wrapped_output.shape}")
print(f"Outputs match: {torch.allclose(original_output, wrapped_output)}")

In [ ]:
# Test text generation
input_ids = torch.randint(0, model_config['vocab_size'], (1, 10))
attention_mask = torch.ones_like(input_ids)

# Generate text with wrapped model
with torch.no_grad():
    generated_ids = wrapped_model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=20,
        temperature=0.7,
        do_sample=True,
        top_k=50,
        top_p=0.9
    )

print(f"Input shape: {input_ids.shape}")
print(f"Generated shape: {generated_ids.shape}")
print(f"New tokens generated: {generated_ids.shape[1] - input_ids.shape[1]}")

## 7. Summary

In this notebook, we've demonstrated how to:

1. Check if a model is compatible with Flask applications and Apple's CoreML format
2. Identify and address compatibility issues
3. Prepare the model for deployment using a compatibility wrapper
4. Save the model in formats suitable for both Flask and CoreML
5. Test the model inference to ensure it works correctly

By following these steps, you can ensure that your model will work seamlessly with both Flask applications and Apple's CoreML format.

## 8. Next Steps

With a compatible model, you can now proceed to:

1. **Flask Deployment**: Use the saved model in a Flask application
2. **CoreML Conversion**: Convert the saved model to CoreML format using coremltools

These steps are covered in separate notebooks:
- `flask_deployment.ipynb`: Deploying the model as a Flask application
- `coreml_conversion.ipynb`: Converting the model to CoreML format